# Ingestion d'un fichier CSV

In [ ]:
# charger les fichiers CSV
import pandas as pd

df = pd.read_csv("../data/csv_excel_files/tickets_support.csv", sep=",")
df.head()

## Convertir une ligne en texte

In [ ]:
def row_to_text(row):
    return "\n".join([f"{col}: {row[col]}" for col in row.index]) 

In [ ]:
for rows in df.iterrows():
    text = row_to_text(rows[1])
    print(text)

## Transformer tout le CSV en Document RAG

In [ ]:
from langchain_core.documents import Document
docs = []

for index, row in df.iterrows():
    text = row_to_text(row)
    metadata = {"source": "tickets_support.csv", "row_index": index}
    doc = Document(page_content=text, metadata=metadata)
    docs.append(doc)

print(f"Nombre de documents créés : {len(docs)}")
print("==============================================================")
print(f"Exemple de document : \n {docs[0].page_content}")
print("==============================================================")
print(f"Exemple de métadonnées : \n {docs[0].metadata}")

In [ ]:
for i, d in enumerate(docs[:2]):
    print("="*50)
    print("DOCUMENT", i)
    print("="*50)
    print(d.page_content, "\n")

# Chunker un document CSV

## Importer le splitter + Application du splitter sur le document CSV

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
)

csv_chunks = splitter.split_documents(docs)

In [ ]:
len(csv_chunks)

In [ ]:
csv_chunks[1].page_content

# Ingestion Excel

In [ ]:
# Lire les feuilles du fichier Excel
import pandas as pd

xls = pd.ExcelFile("../data/csv_excel_files/support_data_advanced.xlsx")

xls.sheet_names

In [ ]:
df_xls1 = pd.read_excel(xls, sheet_name=xls.sheet_names[0])
df_xls2 = pd.read_excel(xls, sheet_name=xls.sheet_names[1])

## Convertir uen feuille Excel en document RAG

In [ ]:
docs_xls1 = [Document(page_content=row_to_text(row), metadata={"source": "excel", "sheet": "tickets_support", "row_index": index}) for index, row in df_xls1.iterrows()]
docs_xls2 = [Document(page_content=row_to_text(row), metadata={"source": "excel", "sheet": "support_data", "row_index": index}) for index, row in df_xls2.iterrows()]

print(f"Nombre de documents créés à partir de la feuille 1 : {len(docs_xls1)}")
print(f"Nombre de documents créés à partir de la feuille 2 : {len(docs_xls2)}")

print("contenu d'un document de la feuille 1 :")
print(docs_xls1[0].page_content)
print("==============================================================")

print("contenu d'un document de la feuille 2 :")
print(docs_xls2[0].page_content)
print("==============================================================")


# Nettoyage d'un fichier CSV

## Charger les fichiers CSV

In [ ]:
import pandas as pd

df = pd.read_csv("../data/csv_excel_files/support_dirty.csv", sep=",")
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

# Création d'une classe professionnel pour appliquer le nettoyage sur un fichier CSV

In [ ]:
class ingest_csv_excel:
    def __init__(self):
        # Standardisation des priorités
        self.priority_map = {
            "high": "Haute",
            "low": "Basse",
            "critical": "Critique",
            "n/a": None,
            "na": None,
        }

    def clean_dates(self, df, col):
        def parse(value):
            try:
                return pd.to_datetime(value, errors="raise")
            except (TypeError, ValueError):
                return pd.NaT

        df[col] = df[col].apply(parse)
        return df

    def clean_priority(self, df, col):
        df[col] = df[col].astype(str).str.lower().map(self.priority_map)
        return df

    def clean_text(self, df, col):
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        return df

    def clean(self, df):
        df = df.copy()

        # 1) Nettoyer les dates
        df = self.clean_dates(df, "creation_date")

        # 2) Normaliser la priorité
        df = self.clean_priority(df, "priority")

        # 3) Nettoyer la description
        df = self.clean_text(df, "description")

        # 4) Supprimer les doublons
        df = df.drop_duplicates()

        # 5) Supprimer les lignes inutilisables
        df = df.dropna(subset=["client", "priority", "description"])
        return df


In [ ]:
cleaner = ingest_csv_excel()

In [ ]:
df_cleaned = cleaner.clean(df)
df_cleaned

# Convertion du tableau propre en document RAG

## Transformation du DataFrame propore en document RAG

In [ ]:
from langchain_core.documents import Document
docs_final = []

for index, row in df_cleaned.iterrows():
    text = row_to_text(row)
    metadata = {"source": "support_dirty", "row_index": index}
    doc = Document(page_content=text, metadata=metadata)
    docs_final.append(doc)

print(f"Nombre de documents créés à partir du DataFrame nettoyé : {len(docs_final)}")

print("contenu d'un document du DataFrame nettoyé :")
print(docs_final[0].page_content)


In [ ]:
# chunker les documents nettoyés
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)
final_chunks = splitter.split_documents(docs_final)

print(f"Nombre de chunks créés à partir des documents nettoyés : {len(final_chunks)}")
print("contenu d'un chunk du DataFrame nettoyé :")
print(final_chunks[0].page_content)